# Ensemble Learning Workflow

This notebook compares independent classifiers, hard voting, manual stacking, and Scikit-Learn's built-in stacking workflow on MNIST.

## Workflow Overview

**Objective:** combine multiple classifiers to improve robustness and generalization.

**Dataset:** MNIST digit classification.

**Primary metric:** accuracy on validation and test splits.

**Validation strategy:** base models are selected on validation data; final stacking can use internal cross-validation to train the blender.

**Production mindset:** compare ensemble accuracy against individual model accuracy and weigh gains against extra inference cost and complexity.

## Stage 0 - Environment and Runtime Controls

Set imports, reproducibility constants, and dataset-size controls for fast or full MNIST runs.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Full MNIST ensemble runs can be slow. Set this to True for the exact full split.
RUN_EXPENSIVE_MNIST = False
MNIST_TRAIN_SIZE = 50_000 if RUN_EXPENSIVE_MNIST else 3_000
MNIST_VALID_SIZE = 10_000 if RUN_EXPENSIVE_MNIST else 1_000
MNIST_TEST_SIZE = 10_000 if RUN_EXPENSIVE_MNIST else 1_000
N_ESTIMATORS = 100 if RUN_EXPENSIVE_MNIST else 50


## Stage 1 - Load and Split MNIST

Create train, validation, and test partitions for ensemble model comparison.

The full benchmark split is:

```text
50,000 training
10,000 validation
10,000 test
```

By default this notebook uses smaller subsets so it runs quickly: 3,000 train, 1,000 validation, and 1,000 test rows. Set `RUN_EXPENSIVE_MNIST = True` in the setup cell for the full benchmark run.

In [2]:
mnist = fetch_openml("mnist_784", as_frame=False, parser="auto")
X_mnist = mnist.data.astype(np.float32)
y_mnist = mnist.target.astype(np.uint8)

train_end = MNIST_TRAIN_SIZE
valid_end = MNIST_TRAIN_SIZE + MNIST_VALID_SIZE
test_end = valid_end + MNIST_TEST_SIZE

X_train = X_mnist[:train_end]
y_train = y_mnist[:train_end]
X_valid = X_mnist[train_end:valid_end]
y_valid = y_mnist[train_end:valid_end]
X_test = X_mnist[valid_end:test_end]
y_test = y_mnist[valid_end:test_end]

X_train.shape, X_valid.shape, X_test.shape

((3000, 784), (1000, 784), (1000, 784))

## Stage 2 - Hard Voting Ensemble

Train diverse base classifiers and combine their predicted labels using majority vote.

Train several different classifiers, compare them on the validation set, then combine them with a voting classifier.

We use:

- Random Forest
- Extra-Trees
- Linear SVM trained with SGD

Since this linear SVM-style classifier does not provide probabilities by default, we use **hard voting**.

In [3]:
rf_clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
extra_trees_clf = ExtraTreesClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
svm_clf = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="hinge", max_iter=1_000, tol=1e-3, random_state=RANDOM_STATE, n_jobs=-1),
)

classifiers = {
    "random_forest": rf_clf,
    "extra_trees": extra_trees_clf,
    "linear_svc": svm_clf,
}

validation_results = []
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    valid_pred = clf.predict(X_valid)
    validation_results.append({
        "model": name,
        "validation_accuracy": accuracy_score(y_valid, valid_pred),
    })

validation_results = pd.DataFrame(validation_results).sort_values("validation_accuracy", ascending=False)
validation_results

,model,validation_accuracy
1,extra_trees,0.929
0,random_forest,0.918
2,linear_svc,0.897


In [4]:
voting_clf = VotingClassifier(
    estimators=[
        ("random_forest", rf_clf),
        ("extra_trees", extra_trees_clf),
        ("linear_svc", svm_clf),
    ],
    voting="hard",
    n_jobs=-1,
)

voting_clf.fit(X_train, y_train)

voting_valid_pred = voting_clf.predict(X_valid)
voting_valid_accuracy = accuracy_score(y_valid, voting_valid_pred)
voting_valid_accuracy

0.93

In [5]:
test_results = []
for name, clf in classifiers.items():
    test_pred = clf.predict(X_test)
    test_results.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, test_pred),
    })

voting_test_pred = voting_clf.predict(X_test)
voting_test_accuracy = accuracy_score(y_test, voting_test_pred)
test_results.append({"model": "hard_voting_ensemble", "test_accuracy": voting_test_accuracy})

pd.DataFrame(test_results).sort_values("test_accuracy", ascending=False)

,model,test_accuracy
3,hard_voting_ensemble,0.927
1,extra_trees,0.924
0,random_forest,0.919
2,linear_svc,0.886


## Stage 3 - Stacking Ensemble

Build prediction features from base models, train a blender, and compare manual stacking with `StackingClassifier`.

Manual stacking process:

1. Train base classifiers on the training set.
2. Predict the validation set with each base classifier.
3. Use these predictions as new features.
4. Train a blender on those new features and the validation labels.
5. For the test set, get base predictions first, then feed them to the blender.

This is a stacking ensemble.

In [6]:
base_classifiers = {
    "random_forest": rf_clf,
    "extra_trees": extra_trees_clf,
    "linear_svc": svm_clf,
}

# The base classifiers were already fit above, but this loop keeps the cell self-contained.
for clf in base_classifiers.values():
    clf.fit(X_train, y_train)

X_valid_predictions = np.column_stack([
    clf.predict(X_valid) for clf in base_classifiers.values()
])
X_test_predictions = np.column_stack([
    clf.predict(X_test) for clf in base_classifiers.values()
])

X_valid_predictions[:5], X_valid_predictions.shape

(array([[9, 0, 9],
        [0, 0, 0],
        [9, 9, 9],
        [8, 8, 8],
        [1, 1, 1]], dtype=uint8),
 (1000, 3))

In [8]:
blender = RandomForestClassifier(n_estimators=2 * N_ESTIMATORS, max_leaf_nodes=40, random_state=RANDOM_STATE, n_jobs=-1)
blender.fit(X_valid_predictions, y_valid)

stacking_test_pred = blender.predict(X_test_predictions)
manual_stacking_accuracy = accuracy_score(y_test, stacking_test_pred)

pd.Series({
    "manual_stacking_test_accuracy": manual_stacking_accuracy,
    "hard_voting_test_accuracy": voting_test_accuracy,
})

manual_stacking_test_accuracy    0.918
hard_voting_test_accuracy        0.927
dtype: float64

In [11]:
stacking_clf = StackingClassifier(
    estimators=[
        ("random_forest", RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)),
        ("extra_trees", ExtraTreesClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)),
        ("linear_svc", make_pipeline(
            StandardScaler(),
            SGDClassifier(loss="hinge", max_iter=1_000, tol=1e-3, random_state=RANDOM_STATE, n_jobs=-1),
        )),
    ],
    final_estimator=LogisticRegression(max_iter=10_000, random_state=RANDOM_STATE),
    cv=3,
    n_jobs=-1,
)

X_train_full = np.concatenate([X_train, X_valid], axis=0)
y_train_full = np.concatenate([y_train, y_valid], axis=0)

stacking_clf.fit(X_train_full, y_train_full)
stacking_test_pred_builtin = stacking_clf.predict(X_test)
stacking_test_accuracy_builtin = accuracy_score(y_test, stacking_test_pred_builtin)

pd.Series({
    "train_full_size": len(X_train_full),
    "built_in_stacking_test_accuracy": stacking_test_accuracy_builtin,
    "manual_stacking_test_accuracy": manual_stacking_accuracy,
    "hard_voting_test_accuracy": voting_test_accuracy,
})

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


train_full_size                    4000.000
built_in_stacking_test_accuracy       0.939
manual_stacking_test_accuracy         0.918
hard_voting_test_accuracy             0.927
dtype: float64

In [10]:
pd.DataFrame({
    "true_label": y_test[:20],
    "voting_prediction": voting_test_pred[:20],
    "manual_stacking_prediction": stacking_test_pred[:20],
    "builtin_stacking_prediction": stacking_test_pred_builtin[:20],
})

,true_label,voting_prediction,manual_stacking_prediction,builtin_stacking_prediction
0,7,7,7,7
1,3,3,3,3
2,0,0,0,0
3,1,1,1,1
4,5,5,5,5
5,2,2,2,5
6,8,7,7,9
7,5,5,5,5
8,0,0,0,0
9,0,0,0,0


## Workflow Summary and Next Steps

Summarize ensemble tradeoffs, validation design, and possible extensions such as soft voting and calibrated probabilities.

The ensemble-learning experiments are complete.

Key ideas:

- ensembles help most when individual models make different errors
- hard voting uses predicted classes; soft voting averages probabilities
- bagging, pasting, random forests, and base stacking models parallelize well
- boosting is mostly sequential
- out-of-bag evaluation gives a validation-like score without a separate validation set
- extra-trees add more randomness by using random thresholds
- stacking trains a blender on predictions from base models